# Modelo genérico · Detecção de anomalias por densidade gaussiana

**Template reutilizável**: estima uma distribuição gaussiana para os dados "normais", calcula a densidade $p(x)$ de cada exemplo, escolhe o limiar ε pelo F1 num conjunto de validação rotulado e marca como anomalia o que fica abaixo dele.

| | |
|---|---|
| **Modelo no sistema** | Machine Learning › 3. Aprendizado não supervisionado › cartão *Gaussian Mixture Model (GMM)* (`/modelar/aprendizado-nao-supervisionado/#m-gmm`) |
| **Por que esse cartão** | O modelo do original é uma densidade gaussiana com features independentes, que é exatamente um GMM com **1 componente** e covariância **diagonal** (`GaussianMixture(n_components=1, covariance_type="diag")`); a seção 5 prova a equivalência. O cartão usa a log-densidade do GMM para anomalias e o generaliza para K componentes, feito na seção 8. Relacionados: *Z-score* (é o caso univariado) e *Isolation Forest* |
| **Origem** | `C3_W1_Anomaly_Detection.ipynb` (*Machine Learning Specialization*, DeepLearning.AI e Stanford Online; servidores: vazão e latência; depois 11 features), revisado, corrigido e generalizado |
| **Tipo de problema** | Não supervisionado (ajuste) com validação semissupervisionada (rótulos só para escolher ε) |
| **Dependências** | `numpy`, `pandas`, `matplotlib`, `scikit-learn` |

### Como usar
1. Edite apenas a célula **1. Configuração** (`>>> ADAPTAR`).
2. **Com rótulos** (`COLUNA_ROTULO`): o ajuste usa só exemplos normais; validação e teste misturam normais e anomalias.
   **Sem rótulos** (`COLUNA_ROTULO = None`): ajusta em tudo e define ε pela fração esperada de anomalias (`TAXA_CONTAMINACAO`).
3. Features muito assimétricas: liste-as em `TRANSFORMAR_LOG` (o modelo supõe forma aproximadamente gaussiana).

### O que mudou em relação ao notebook original
- **Estabilidade numérica:** com 11 features o ε escolhido já era 1,4 × 10⁻¹⁸; com mais dimensões, `p(x)` chega a zero no `float64` e todos os pontos empatam. Agora tudo é calculado em **log-densidade**, e o ε é buscado na escala log.
- `select_threshold` dividia por zero (`tp + fp = 0`) nos limiares baixos, gerando avisos e `nan`; agora esses casos têm F1 = 0. Em empates do F1 máximo, o original ficava com o menor ε (colado nas anomalias da validação); agora fica o do meio do intervalo empatado.
- `multivariate_gaussian` e `visualize_fit` vinham de `utils.py` (ausente) e foram reescritos; a covariância pode ser diagonal (como no original) ou completa (`TIPO_COVARIANCIA = "full"`, que captura correlação entre features).
- O original escolhia ε e media o F1 **no mesmo conjunto de validação**. Agora há um conjunto de **teste** separado, com precisão, recall, F1 e PR-AUC (métricas do cartão).
- Sem as 2 features nomeadas (vazão, latência), os eixos fixos `[0, 30]` e as funções `load_data()` / `load_data_multi()` do dataset original.

## 1. Configuração

In [ ]:
# >>> ADAPTAR: única célula que precisa ser editada para usar outro dataset.

# --- dados ---
CAMINHO_DADOS = None        # ex.: "dados/sensores.csv"; None = dados sintéticos
SEPARADOR = None
TEM_CABECALHO = True        # False se a primeira linha já é dado
NOMES_COLUNAS = None        # nomes quando TEM_CABECALHO = False; sem nomes, viram coluna_0, coluna_1, …
COLUNAS_FEATURES = None     # None = todas as colunas numéricas, exceto o rótulo
COLUNA_ROTULO = "anomalia"  # coluna 0/1 (1 = anomalia); None se não houver rótulo
VALOR_ANOMALIA = 1          # valor da coluna de rótulo que indica anomalia
TRANSFORMAR_LOG = []        # colunas assimétricas a transformar com log1p; ex.: ["tempo_resposta"]

# --- modelo ---
TIPO_COVARIANCIA = "diag"   # "diag" = features independentes (original); "full" = com correlações
TAXA_CONTAMINACAO = 0.01    # sem rótulo: fração esperada de anomalias (define ε)
MAX_COMPONENTES = 5         # seção 8: testa GMMs de 1 a MAX_COMPONENTES componentes

# --- divisão e reprodutibilidade ---
FRACAO_TREINO_NORMAIS = 0.6  # com rótulo: fração dos normais usada no ajuste; o resto vai para validação e teste
SEMENTE = 42

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.mixture import GaussianMixture

rng = np.random.default_rng(SEMENTE)

## 2. Carregamento e divisão

In [ ]:
def carregar_tabela(caminho, separador=None, tem_cabecalho=True, nomes_colunas=None, planilha=0):
    """Lê CSV/TXT/TSV, Excel, Parquet ou JSON e devolve um DataFrame."""
    caminho = Path(caminho)
    if not caminho.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {caminho.resolve()}")
    ext = caminho.suffix.lower()
    cabecalho = 0 if tem_cabecalho else None
    if ext in {".csv", ".txt", ".tsv"}:
        sep = separador if separador is not None else ("\t" if ext == ".tsv" else None)
        df = pd.read_csv(caminho, sep=sep, engine="python", header=cabecalho)
    elif ext in {".xlsx", ".xls"}:
        df = pd.read_excel(caminho, sheet_name=planilha, header=cabecalho)
    elif ext == ".parquet":
        df = pd.read_parquet(caminho)
    elif ext == ".json":
        df = pd.read_json(caminho)
    else:
        raise ValueError(f"Formato não suportado: {ext}")
    if nomes_colunas is not None:
        df.columns = nomes_colunas
    elif not tem_cabecalho:
        df.columns = [f"coluna_{i}" for i in range(df.shape[1])]
    return df


def dados_sinteticos(n_normais=600, n_anomalias=18, n_features=2, semente=SEMENTE):
    """Normais: gaussiana com features levemente correlacionadas. Anomalias: espalhadas ao redor."""
    r = np.random.default_rng(semente)
    A = r.normal(size=(n_features, n_features)) * 0.3 + np.eye(n_features)
    normais = r.normal(size=(n_normais, n_features)) @ A + 15
    anomalias = 15 + r.choice([-1, 1], size=(n_anomalias, n_features)) * r.uniform(3.5, 7, size=(n_anomalias, n_features))
    X = np.vstack([normais, anomalias])
    df = pd.DataFrame(X, columns=[f"feature_{i + 1}" for i in range(n_features)])
    df["anomalia"] = np.r_[np.zeros(n_normais, int), np.ones(n_anomalias, int)]
    return df.sample(frac=1, random_state=semente).reset_index(drop=True)


df = (dados_sinteticos() if CAMINHO_DADOS is None
      else carregar_tabela(CAMINHO_DADOS, SEPARADOR, TEM_CABECALHO, NOMES_COLUNAS))
features = COLUNAS_FEATURES or [c for c in df.select_dtypes(include="number").columns if c != COLUNA_ROTULO]
df = df.dropna(subset=features)
for c in TRANSFORMAR_LOG:
    if (df[c] < 0).any():
        raise ValueError(f"{c} tem valores negativos; log1p não se aplica.")
    df[c] = np.log1p(df[c])
X_all = df[features].to_numpy(dtype=float)
print("formato:", X_all.shape, "| features:", features)

if COLUNA_ROTULO is not None:
    rotulo = (df[COLUNA_ROTULO] == VALOR_ANOMALIA).astype(int).to_numpy()
    idx_norm, idx_anom = rng.permutation(np.flatnonzero(rotulo == 0)), rng.permutation(np.flatnonzero(rotulo == 1))
    n_tr = int(FRACAO_TREINO_NORMAIS * len(idx_norm))
    resto_norm = idx_norm[n_tr:]
    i_val = np.r_[resto_norm[: len(resto_norm) // 2], idx_anom[: len(idx_anom) // 2]]
    i_te = np.r_[resto_norm[len(resto_norm) // 2:], idx_anom[len(idx_anom) // 2:]]
    X_train = X_all[idx_norm[:n_tr]]
    X_val, y_val = X_all[i_val], rotulo[i_val]
    X_test, y_test = X_all[i_te], rotulo[i_te]
    print(f"ajuste: {len(X_train)} normais | validação: {len(y_val)} ({y_val.sum()} anomalias) | "
          f"teste: {len(y_test)} ({y_test.sum()} anomalias)")
    if y_val.sum() == 0 or y_test.sum() == 0:
        raise ValueError("São necessárias anomalias rotuladas na validação e no teste.")
else:
    X_train, X_val, y_val, X_test, y_test = X_all, None, None, None, None
    print("sem rótulos: ajuste em todos os exemplos; ε pela TAXA_CONTAMINACAO")

In [ ]:
# Forma de cada feature: o modelo supõe algo próximo de uma gaussiana.
# Assimetria forte → inclua a coluna em TRANSFORMAR_LOG.
n = len(features)
fig, eixos = plt.subplots(1, min(n, 6), figsize=(3.2 * min(n, 6), 2.8), squeeze=False)
for ax, j in zip(eixos[0], range(min(n, 6))):
    ax.hist(X_train[:, j], bins=40); ax.set_title(features[j], fontsize=9)
plt.tight_layout(); plt.show()
display(pd.DataFrame(X_train, columns=features).skew().rename("assimetria").to_frame().T.round(2))

## 3. Estimação dos parâmetros
Para cada feature $j$: média $\mu_j = \frac{1}{m}\sum_i x_j^{(i)}$ e variância $\sigma_j^2 = \frac{1}{m}\sum_i (x_j^{(i)} - \mu_j)^2$ (estimadores de máxima verossimilhança, divididos por $m$, como no original e no GMM).

Com covariância completa, $\Sigma = \frac{1}{m}\sum_i (\mathbf{x}^{(i)} - \boldsymbol\mu)(\mathbf{x}^{(i)} - \boldsymbol\mu)^\top$.

In [ ]:
def estimate_gaussian(X):
    """Média e variância de cada feature. X: (m, n) → mu: (n,), var: (n,)."""
    m = X.shape[0]
    mu = X.sum(axis=0) / m
    var = ((X - mu) ** 2).sum(axis=0) / m
    return mu, var


def estimate_covariance(X):
    mu = X.mean(axis=0)
    D = X - mu
    return D.T @ D / X.shape[0]


mu, var = estimate_gaussian(X_train)
Sigma = np.diag(var) if TIPO_COVARIANCIA == "diag" else estimate_covariance(X_train)
display(pd.DataFrame({"média": mu, "variância": var}, index=features).round(4))

# verificação com o exemplo numérico do original (equivalente vetorizado)
Xt = np.array([[1., 2.], [3., 4.], [5., 9.]])
assert np.allclose(estimate_gaussian(Xt), (Xt.mean(0), Xt.var(0)))

## 4. Densidade gaussiana em escala log

$$p(\mathbf{x}) = \frac{1}{(2\pi)^{n/2}\lvert\Sigma\rvert^{1/2}}\exp\!\left(-\tfrac12(\mathbf{x}-\boldsymbol\mu)^\top\Sigma^{-1}(\mathbf{x}-\boldsymbol\mu)\right)$$

Com $\Sigma$ diagonal, é o produto das densidades de cada feature, $p(\mathbf{x}) = \prod_j p(x_j;\mu_j,\sigma_j^2)$, e $\log p(\mathbf{x}) = \text{const} - \tfrac12\sum_j z_j^2$: a soma dos **z-scores ao quadrado**. Trabalhar com $\log p$ evita que produtos de números muito pequenos virem zero.

In [ ]:
def log_multivariate_gaussian(X, mu, Sigma):
    """log p(x) para cada linha de X, com Sigma (n, n). Usa Cholesky: estável e sem inversa explícita."""
    n = X.shape[1]
    L = np.linalg.cholesky(Sigma)
    Z = np.linalg.solve(L, (X - mu).T)              # L·Z = (x − mu)ᵀ
    mahal = (Z ** 2).sum(axis=0)
    log_det = 2 * np.log(np.diag(L)).sum()
    return -0.5 * (n * np.log(2 * np.pi) + log_det + mahal)


def multivariate_gaussian(X, mu, var):
    """Versão em densidade (como no original). var (n,) = diagonal; (n, n) = covariância completa."""
    S = np.diag(var) if np.ndim(var) == 1 else var
    return np.exp(log_multivariate_gaussian(X, mu, S))


logp_train = log_multivariate_gaussian(X_train, mu, Sigma)
print("log p(x) no ajuste: min", logp_train.min().round(2), "| mediana", np.median(logp_train).round(2))

## 5. Equivalência com o cartão: GMM de 1 componente
Se a implementação estiver certa, a log-densidade é a mesma do `GaussianMixture` com um componente e o mesmo tipo de covariância.

In [ ]:
gmm1 = GaussianMixture(n_components=1, covariance_type=TIPO_COVARIANCIA, reg_covar=1e-12,
                       random_state=SEMENTE).fit(X_train)
dif = np.max(np.abs(gmm1.score_samples(X_train) - logp_train))
print(f"máx. |log p (do zero) − log p (GaussianMixture)| = {dif:.2e}")
assert dif < 1e-6, "implementação diverge do GaussianMixture"

## 6. Escolha do limiar ε
Um exemplo é anomalia se $p(x) < \varepsilon$, isto é, $\log p(x) < \log\varepsilon$. Com rótulos, ε é o que maximiza o $F_1$ na **validação**:

$$\text{prec} = \frac{tp}{tp+fp},\qquad \text{rec} = \frac{tp}{tp+fn},\qquad F_1 = \frac{2\cdot\text{prec}\cdot\text{rec}}{\text{prec}+\text{rec}}$$

Como no original, a busca percorre 1000 passos entre o menor e o maior valor; aqui, na escala log.

In [ ]:
def select_threshold(y_val, score_val, passos=1000):
    """Escolhe o limiar que maximiza o F1; anomalia = score < limiar.
    score = log p(x) (ou p(x), como no original). Devolve (melhor_limiar, melhor_F1).
    Empates: o original ficava com o primeiro (o menor) limiar; aqui fica o do meio do intervalo empatado,
    que deixa folga dos dois lados e generaliza melhor."""
    candidatos = np.linspace(score_val.min(), score_val.max(), passos)[1:]
    f1s = np.zeros(len(candidatos))
    for k, eps in enumerate(candidatos):
        pred = score_val < eps
        tp = np.sum(pred & (y_val == 1)); fp = np.sum(pred & (y_val == 0)); fn = np.sum(~pred & (y_val == 1))
        if tp == 0:                                      # sem verdadeiros positivos: F1 = 0 (evita 0/0)
            continue
        prec, rec = tp / (tp + fp), tp / (tp + fn)
        f1s[k] = 2 * prec * rec / (prec + rec)
    melhores = np.flatnonzero(np.isclose(f1s, f1s.max()))
    return candidatos[melhores[len(melhores) // 2]], float(f1s.max())


if COLUNA_ROTULO is not None:
    logp_val = log_multivariate_gaussian(X_val, mu, Sigma)
    log_eps, F1_val = select_threshold(y_val, logp_val)
    print(f"melhor ε na validação: {np.exp(log_eps):.3e} (log ε = {log_eps:.3f}) | F1 validação: {F1_val:.4f}")
else:
    log_eps = np.quantile(logp_train, TAXA_CONTAMINACAO)
    print(f"ε pela contaminação de {TAXA_CONTAMINACAO:.1%}: {np.exp(log_eps):.3e} (log ε = {log_eps:.3f})")
print("anomalias marcadas no conjunto de ajuste:", int((logp_train < log_eps).sum()))

In [ ]:
# Com duas features: curvas de nível da densidade e anomalias circuladas (a figura do original).
if X_train.shape[1] == 2:
    X_plot = X_all
    g1, g2 = np.meshgrid(np.linspace(X_plot[:, 0].min() - 1, X_plot[:, 0].max() + 1, 300),
                         np.linspace(X_plot[:, 1].min() - 1, X_plot[:, 1].max() + 1, 300))
    LP = log_multivariate_gaussian(np.c_[g1.ravel(), g2.ravel()], mu, Sigma).reshape(g1.shape)
    plt.figure(figsize=(6, 5))
    plt.contour(g1, g2, LP, levels=np.sort(log_eps + np.arange(-12, 13, 3)), linewidths=0.7)
    plt.scatter(X_plot[:, 0], X_plot[:, 1], marker="x", s=12)
    marc = log_multivariate_gaussian(X_plot, mu, Sigma) < log_eps
    plt.plot(X_plot[marc, 0], X_plot[marc, 1], "o", ms=10, mfc="none", mew=2, label="anomalia (p < ε)")
    plt.xlabel(features[0]); plt.ylabel(features[1]); plt.legend(); plt.title("Densidade estimada e anomalias")
    plt.show()

## 7. Avaliação no teste

In [ ]:
if COLUNA_ROTULO is not None:
    logp_te = log_multivariate_gaussian(X_test, mu, Sigma)
    pred_te = (logp_te < log_eps).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred_te, labels=[0, 1]).ravel()
    display(pd.Series({"precisão": precision_score(y_test, pred_te, zero_division=0), "recall": recall_score(y_test, pred_te),
                       "F1": f1_score(y_test, pred_te), "PR-AUC": average_precision_score(y_test, -logp_te),
                       "VP": tp, "FP": fp, "FN": fn, "VN": tn}, name="teste").to_frame().T.round(4))
else:
    print("Sem rótulos não há métrica objetiva: revise uma amostra dos exemplos marcados com a área de negócio.")
    display(df.loc[logp_train < log_eps, features].head(10))

## 8. Além de uma gaussiana: GMM com K componentes
Quando os dados normais formam vários grupos, uma única gaussiana superestima a densidade entre eles. O cartão escolhe K pelo **BIC** (menor é melhor) e usa `score_samples` como log-densidade. O limiar é escolhido de novo, na validação.

In [ ]:
bics = {}
for k in range(1, MAX_COMPONENTES + 1):
    g = GaussianMixture(n_components=k, covariance_type=TIPO_COVARIANCIA, n_init=3, random_state=SEMENTE).fit(X_train)
    bics[k] = (g.bic(X_train), g)
k_bic = min(bics, key=lambda k: bics[k][0])
display(pd.Series({k: v[0] for k, v in bics.items()}, name="BIC").to_frame().T.round(1))
print("K pelo BIC:", k_bic)
gmmK = bics[k_bic][1]
if COLUNA_ROTULO is not None:
    eps_K, F1_K = select_threshold(y_val, gmmK.score_samples(X_val))
    s_te = gmmK.score_samples(X_test)
    print(f"GMM K = {k_bic}: F1 teste {f1_score(y_test, s_te < eps_K):.4f} | PR-AUC teste {average_precision_score(y_test, -s_te):.4f}")
    print(f"gaussiana única: F1 teste {f1_score(y_test, pred_te):.4f} | PR-AUC teste {average_precision_score(y_test, -logp_te):.4f}")

## 9. Pontuar novos dados

In [ ]:
novos = df[features].sample(5, random_state=SEMENTE)     # >>> ADAPTAR: mesmas colunas (já com TRANSFORMAR_LOG aplicado)
lp = log_multivariate_gaussian(novos.to_numpy(dtype=float), mu, Sigma)
display(novos.assign(log_p=lp.round(3), anomalia=lp < log_eps))

## 10. Próximos passos (segundo os cartões do sistema)
- Detecção de anomalias × classificação supervisionada: com **muitos** exemplos rotulados das duas classes, um classificador tende a ser melhor; a abordagem por densidade brilha quando as anomalias são raras e variadas.
- Escolha de features: compare os valores de cada feature nas anomalias que o modelo não pega; crie razões ou combinações que as destaquem.
- **Prefira Isolation Forest** em alta dimensionalidade ou grande volume; **LOF** para anomalias locais; **z-score robusto** para uma métrica por vez.